In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

torch.manual_seed(42)
torch.set_printoptions(precision=4, sci_mode=False)

# 데이터
data = [
    ("i love the movie",1),
    ("this film is great",1),
    ("i really like this",1),
    ("this is amazing",1),
    ("i hate this movie",0),
    ("this film is terrible",0),
    ("this is boring",0)
]
#지도학습 -> 분류-> 라벨이 잘못 달려있으면 이상하게 학습함.

# 토큰화 및 vocabulary 생성
def tokenize(text):
    return text.lower().split()
# 텍스트를 소문자로 바꾸고 공백 기준으로 나누어 단어 리스트(토큰)으로 만들.ㅁ

special_tokens = ["<PAD>", "<UNK>"]
# 문장 길이를 맞추기 위한 <PAD>, 모르는 단어 처리를 위한 <UNK> 특수 토큰을 정의

word_set = set()
for sentence, label in data:
    word_set.update(tokenize(sentence))
# 중복을 허용하지 않는 집합(set)을 생성하여 전체 데이터의 고유 단어를 수집.

vocab = special_tokens + sorted(list(word_set))
# 특수 토큰과 알파벳순로 정렬된 단어들을 합쳐 최종 단어 사전(vocab)을 만듦.
word2idx = {word:idx for idx, word in enumerate(vocab)}
# 단어를 넣으면 숫자로 나오는 "단어-번호" 딕셔너리를 만듦(인코더용)
idx2word = {idx:word for word, idx in word2idx.items()}
# 숫자를 넣으면 단어가 나오는 "번호-단어" 딕셔너리를 만듦(디코더용)

pad_idx = word2idx["<PAD>"]
unk_idx = word2idx["<UNK>"]
#나중에 패딩 처리에 사용할 <PAD>와 <UNK>의 고유 번호를 변수에 저장

print("vocab:", vocab)
print("vocab size:", len(vocab))

 # 문장을 인덱스로 반환
max_len = max(len(tokenize(sentence)) for sentence, _ in data)
# 모든 문장 중 가장 긴 뭊앙의 단어 개수를 찾아 기준(max_len)

def encode(sentence):
    tokens = tokenize(sentence)
    # 입력받은 문장을 단위 단위로 쪼갬(토큰화)
    ids = [word2idx.get(token, unk_idx) for token in tokens]
    # 단어를 사전에 등록된 숫자로 바꿈. 사전에 없으면 <UNK>로 처리
    ids += [pad_idx] * (max_len - len(ids))
    # 문장 길이를 max_len를 맞추기 위해 부족한 뒷부분을 <PAD> 번호로 부여
    return ids

encoded_data = [(encode(sentence), label) for sentence, label in data]
# 전체 데이터 (문장,라벨)를 돌면서 문장을 숫자 리스트로 변환하여 저장

print("Encoded Samples:")
for i in range(len(encoded_data)):
    print(data[i][0], "->", encoded_data[i][0], "label:",encoded_data[i][1])
# 원본 문장과 변환된 숫자(인덱스) 리스트, 그리고 정답(라벨)을 출력하여 확인.

 # Dataset / DataLoader
class TextDataset(Dataset):
    # 파이토치 전용 Dataset 클래스를 정의
    def __init__(self,encode_data):
        self.encoded_data = encoded_data
        #초기화 단계: 인코딩된 데이터를 클래스 내부 변수에 저장
    def __len__(self):
        return len(self.encoded_data)
        # 데이터셋 전체의 개수를 반환(len(dataset) 호출시 실행
    def __getitem__(self,idx):
        input_ids, label = self.encoded_data[idx]
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)
        # 특정 인덱스(idx)의 데이터를 가져와 파이토치 텐서 형태로 반환
dataset = TextDataset(encoded_data)
# 위에서 만든 클래스를 이용해 실제 데이터셋 객체를 생성
loader = DataLoader(dataset, batch_size=4, shuffle=True)
#데이터 로더: 데이터를 4개씩 묶고(batch_size), 순서를 섞어서(shuffle)

vocab: ['<PAD>', '<UNK>', 'amazing', 'boring', 'film', 'great', 'hate', 'i', 'is', 'like', 'love', 'movie', 'really', 'terrible', 'the', 'this']
vocab size: 16
Encoded Samples:
i love the movie -> [7, 10, 14, 11] label: 1
this film is great -> [15, 4, 8, 5] label: 1
i really like this -> [7, 12, 9, 15] label: 1
this is amazing -> [15, 8, 2, 0] label: 1
i hate this movie -> [7, 6, 15, 11] label: 0
this film is terrible -> [15, 4, 8, 13] label: 0
this is boring -> [15, 8, 3, 0] label: 0


In [2]:
# Positional Encoding 구현
class PositionalEncoding(nn.Module):
    def __init__(self,d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model) # (max_len, d_model)
        position = torch.arange(0, max_len,dtype=torch.float).unsqueeze(1) #(max_len,1)

        div_term = torch.exp(
            torch.arange(0, d_model,2,dtype=torch.float) * (-math.log(10000.0)/d_model)
        )

        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe",pe)

    def foward(self,x):
        seq_len = x.size(1)
        return x + self.pe[:,:seq_len,:]

In [3]:
# Scaled Dot product Attention

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)

    scores = torch.matmul(Q,K.transpose(-2,-1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.mask_fill(mask==0, float("-inf"))
    attn_weights = torch.softmax(scores,dim=-1)

    output = torch.matmul(attn_weights, V)

    return output, attn_weights

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads==0, "d_model must be divisible by num_heads"

        self.d_model = self.d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        self.out_proj = nn.Linear(d_model,d_model, bias=False)
        
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads==0, "d_model must be divisible by num_heads"

        self.d_model = self.d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        self.out_proj = nn.Linear(d_model,d_model, bias=False)
    def forward(self,x,mask=None):
        batch_size,seq_len,_ = x.shape

        Q = self.W_q
        K = self.W_k
        V = self.W_v

        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)

        attn_output, attn_weights = scaled_dot_product_attention(Q,K,V,mask)

        attn_output = attn_output.transpose(1,2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.d_model)

        output = self.out_proj(attn_output)
        return output, attn_weights

self.out_proj: 각기 다른 head들이 계산한 결과물들을 다시 하나로 합치고 최종적인 특징을 추출하기 위한 출력 선형 변환 층

$$
MultiHead(Q,K,V) = Concat(head_1,head_2,\cdots,head_h)~W^0
$$

여기서, $W^0$가 바로 코드에서의 **self.out_proj**

In [5]:
# Position-wise Feed Forward Network 구현

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        # d_model: 모델의 입력/출력 차원
        # d_ff: 은닉층(Hidden layer)의 차원(일반적으로 d_model의 4배 정도로 설정)
        # 논문의 제시(적당한 효율성), 보통 512 -> 2048로 확장
        # dropout: 과적합 방지를 위한 dropout qldbf
        super().__init__()

        self.net = nn.Sequential( #nn.Sequential을 사용하여 여러 층을 순차적으로 쌓음 
            nn.Linear(d_model,d_ff), # 첫 번째 선형 변환: d_model을 d_ff 크기로 확장 
            nn.ReLU(), # 비선형을 추가하여 복잡한 패턴 학습 가능
            nn.Dropout(dropout), # 학습시 무작위로 뉴런을 꺼서 모델의 일반화 성능 향상
            nn.Linear(d_ff, d_model) #두 번째 선형 변환: d_ff 크기를 다시 원래의 d_model 크기로 축소
        )
    def forward(self,x): # x: (batch_size, seq_len, d_model)
        return self.net(x) #정의된 sequential 네트워크에 데이터를 통과시켜 변환

- 어텐션 매커니즘: 단어 간의 관계(어디를 볼지)를 집중
- Feed-forward: 각 위치의 정보를 개별적으로 깊게 처리하는 역할

- 왜 d_ff 차원을 키웠다가 줄일까요?
	- 단순히 선형 층 하나만 쓰면 데이터 표현력이 제한. 차원을 일시적으로 확장했다가 줄이는 과정에서 활성화 함수를 거치면서 데이터 내의 복잡하고 고차원적인 특징을 추출할 수 있게 됨

In [6]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        # d_model: 입력 데이터의 차원
        # num_heads: 멀티 헤드 어텐션의 사용할 헤드 수
        # d_ff: feed-forward 네트워크의 은닉층 차원
        # dropout: dropout 비율
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        self.ffn = FeedForward(d_model,d_ff,dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (batch_size, seq_len, d_model)
        # mask: 패딩이나 미래 시점의 정보를 가리기 위한 마스크(선택사항)

        # [sub-layer1: Multi-head Attention]
        attn_output, attn_weights = self.mha(x,mask)
        x = self.norm1(x+self.dropout1(attn_output))
        # 잔차 연결(add) 및 레이어 정규화

        # [sub-layer1: Feed-forward Network]
        ffn_output = self.ffn(x)
        x = self.norm2(x+self.dropout2(ffn_output))

        return x, attn_weights

- Seq2seq
    - 입력: 1 5 3 9
    - 출력: 9 3 5 1

In [8]:
import math
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
random.seed(42)

torch.set_printoptions(precision=4, sci_mode=False)

# Vocabulary 정의

speical_tokens = ["<PAD>", "<BOS>","<EOS>"]
number_tokens = [str(i) for i in range(1,10)]

vocab = speical_tokens + number_tokens
token_to_idx = {tok:i for i,tok in enumerate(vocab)}
idx_to_toekn = {i:tok for tok,i in token_to_idx.items()}

PAD_IDX = token_to_idx["<PAD>"]
BOS_IDX = token_to_idx["<BOS>"]
EOS_IDX = token_to_idx["<EOS>"]

print("vocab", vocab)
print("vocab_size", len(vocab))
print("PAD_IDX", PAD_IDX)
print("EOS_IDX", EOS_IDX)
print("BOS_IDX", BOS_IDX)

# 샘플 생성 함수
def generate_sample(min_len=3, max_len=5):
    length = random.randint(min_len, max_len)

    src_tokens = [str(random.randint(1,9)) for _ in range(length)]
    tgt_tokens = list(reversed(src_tokens))

    return src_tokens, tgt_tokens
    
# 샘플 몇개 확인
for _ in range(5):
    src, tgt = generate_sample()
    print("src:", src, "tgt:", tgt)

# 토큰 인코딩 함수
def encode_tokens(tokens):
    return [token_to_idx[t] for t in tokens]
    
# decoder 입력과 정답 만들기

def make_decoder_input_target(tgt_tokens):
    dec_input_tokens = ["<BOS>"] + tgt_tokens
    dec_target_tokens = tgt_tokens + ["<EOS>"]

    dec_input_ids = encode_tokens(dec_input_tokens)
    dec_target_ids = encode_tokens(dec_target_tokens)

    return dec_input_ids, dec_target_ids

# 한 샘플을 인덱스로 확인
src_tokens, tgt_tokens = generate_sample()

src_ids = encode_tokens(src_tokens)
dec_input_ids, dec_target_ids = make_decoder_input_target(tgt_tokens)

print("src_tokens:", src_tokens)
print("tgt_tokens:", tgt_tokens)
print("src_ids", src_ids)
print("dec_input_ids:", dec_input_ids)
print("dec_target_ids:", dec_target_ids) 

class ReverseDataset(Dataset):
    def __init__(self, num_samples=5000,min_len=3, max_len=5):
        self.samples = [generate_sample(min_len,max_len) for _ in range(num_samples)]
    def __len__(self):
        return len(self.samples)
    def __getitem__(self,idx):
        src_tokens, tgt_tokens = self.samples[idx]

        src_ids = encode_tokens(src_tokens)
        dec_input_ids,dec_target_ids = make_decoder_input_target(tgt_tokens)

        return(
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(dec_input_ids, dtype=torch.long),
            torch.tensor(dec_target_ids, dtype=torch.long)
        )
        
# 패딩 함수
## 문장 길이가 서로 다르기 때문에 padding
def pad_sequence(seq, max_len, pad_value):
    return seq + [pad_value]* (max_len - len(seq))
    
# collate 함수
## 배치 단위로 데이터를 받아 길이를 맞춤.

def collate_fn(batch):
    src_list, dec_in_list, dec_tgt_list = zip(*batch)

    src_lengths = [len(x) for x in src_list]
    dec_lengths = [len(x) for x in dec_in_list]

    max_src_len = max(src_lengths)
    max_dec_len = max(dec_lengths)

    padded_src = []
    padded_dec_in = []
    padded_dec_tgt = []

    for src, dec_in, dec_tgt in zip(src_list, dec_in_list, dec_tgt_list):
        padded_src.append(pad_sequence(src.tolist(), max_src_len, PAD_IDX))
        padded_dec_in.append(pad_sequence(dec_in.tolist(), max_dec_len, PAD_IDX))
        padded_dec_tgt.append(pad_sequence(dec_tgt.tolist(), max_dec_len, PAD_IDX))

    return (
        torch.tensor(padded_src, dtype=torch.long),
        torch.tensor(padded_dec_in, dtype=torch.long),
        torch.tensor(padded_dec_tgt, dtype=torch.long),
    )
# DataLoader 만들기
train_dataset = ReverseDataset(num_samples=5000)
valid_dataset = ReverseDataset(num_samples=1000)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

vocab ['<PAD>', '<BOS>', '<EOS>', '1', '2', '3', '4', '5', '6', '7', '8', '9']
vocab_size 12
PAD_IDX 0
EOS_IDX 2
BOS_IDX 1
src: ['2', '1', '5', '4', '4'] tgt: ['4', '4', '5', '1', '2']
src: ['2', '9', '2'] tgt: ['2', '9', '2']
src: ['7', '1', '1', '2', '4'] tgt: ['4', '2', '1', '1', '7']
src: ['9', '1', '9'] tgt: ['9', '1', '9']
src: ['9', '7', '4'] tgt: ['4', '7', '9']
src_tokens: ['5', '1', '3', '7']
tgt_tokens: ['7', '3', '1', '5']
src_ids [7, 3, 5, 9]
dec_input_ids: [1, 9, 5, 3, 7]
dec_target_ids: [9, 5, 3, 7, 2]


In [9]:
# 배치 shape 확인
src, dec_in, dec_tgt = next(iter(train_loader))

print("src shape:", src.shape)
print("dec_in shape:", dec_in.shape)
print("dec_tgt shape:", dec_tgt.shape)

print("src[0]:", src[0])
print("dec_in[0]:", dec_in[0])
print("dec_tgt[0]:", dec_tgt[0])

src shape: torch.Size([64, 5])
dec_in shape: torch.Size([64, 6])
dec_tgt shape: torch.Size([64, 6])
src[0]: tensor([5, 5, 8, 4, 0])
dec_in[0]: tensor([1, 4, 8, 5, 5, 0])
dec_tgt[0]: tensor([4, 8, 5, 5, 2, 0])


In [10]:
# Positional Encoding 구현
class PositionalEncoding(nn.Module):
    def __init__(self,d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model) # (max_len, d_model)
        position = torch.arange(0, max_len,dtype=torch.float).unsqueeze(1) #(max_len,1)

        div_term = torch.exp(
            torch.arange(0, d_model,2,dtype=torch.float) * (-math.log(10000.0)/d_model)
        )

        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe",pe)

    def forward(self,x):
        seq_len = x.size(1)
        return x + self.pe[:,:seq_len,:]

In [11]:
# Scaled Dot product Attention

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)

    scores = torch.matmul(Q,K.transpose(-2,-1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.mask_fill(mask==0, float("-inf"))
    attn_weights = torch.softmax(scores,dim=-1)

    output = torch.matmul(attn_weights, V)

    return output, attn_weights

In [12]:
# 1. self attention / 2. cross attention 둘 다 사용할 수 있게 query,key,value 구현
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        query_len = query.size(1)
        key_len = key.size(1)
        value_len = value.size(1)

        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        Q = Q.view(batch_size, query_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, key_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, value_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, query_len, self.d_model)

        output = self.out_proj(attn_output)
        return output, attn_weights

In [13]:
 # Position-wise Feed Forward Network 구현

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        # d_model: 모델의 입력/출력 차원
        # d_ff: 은닉층(Hidden layer)의 차원(일반적으로 d_model의 4배 정도로 설정)
        # 논문의 제시(적당한 효율성), 보통 512 -> 2048로 확장
        # dropout: 과적합 방지를 위한 dropout qldbf
        super().__init__()

        self.net = nn.Sequential( #nn.Sequential을 사용하여 여러 층을 순차적으로 쌓음 
            nn.Linear(d_model,d_ff), # 첫 번째 선형 변환: d_model을 d_ff 크기로 확장 
            nn.ReLU(), # 비선형을 추가하여 복잡한 패턴 학습 가능
            nn.Dropout(dropout), # 학습시 무작위로 뉴런을 꺼서 모델의 일반화 성능 향상
            nn.Linear(d_ff, d_model) #두 번째 선형 변환: d_ff 크기를 다시 원래의 d_model 크기로 축소
        )
    def forward(self,x): # x: (batch_size, seq_len, d_model)
        return self.net(x) #정의된 sequential 네트워크에 데이터를 통과시켜 변환

In [14]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        # d_model: 입력 데이터의 차원
        # num_heads: 멀티 헤드 어텐션의 사용할 헤드 수
        # d_ff: feed-forward 네트워크의 은닉층 차원
        # dropout: dropout 비율
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        self.ffn = FeedForward(d_model,d_ff,dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (batch_size, seq_len, d_model)
        # mask: 패딩이나 미래 시점의 정보를 가리기 위한 마스크(선택사항)

        # [sub-layer1: Multi-head Attention]
        attn_output, attn_weights = self.mha(x,x,x,mask)
        x = self.norm1(x+self.dropout1(attn_output))
        # 잔차 연결(add) 및 레이어 정규화

        # [sub-layer1: Feed-forward Network]
        ffn_output = self.ffn(x)
        x = self.norm2(x+self.dropout2(ffn_output))

        return x, attn_weights

In [15]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        # d_model: 입력 데이터의 차원
        # num_heads: 멀티 헤드 어텐션의 사용할 헤드 수
        # d_ff: feed-forward 네트워크의 은닉층 차원
        # dropout: dropout 비율
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        self.ffn = FeedForward(d_model,d_ff,dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (batch_size, seq_len, d_model)
        # mask: 패딩이나 미래 시점의 정보를 가리기 위한 마스크(선택사항)

        # [sub-layer1: Multi-head Attention]
        attn_output, attn_weights = self.mha(x,mask)
        x = self.norm1(x+self.dropout1(attn_output))
        # 잔차 연결(add) 및 레이어 정규화

        # [sub-layer1: Feed-forward Network]
        ffn_output = self.ffn(x)
        x = self.norm2(x+self.dropout2(ffn_output))

        return x, attn_weights



In [16]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx, max_len, dropout=0.1):
        # vocab_size: 사전의 크기
        # d_model: 임베딩 및 모델 내부의 차원
        # num_heads: 멀티 헤드 어텐션의 헤드 수
        # d_ff: feed-forward층의 은닉 차원
        # num_layers: 인코더 블록을 쌓을 횟수(ex)bert-base는 12개)
        # pad_idx: 패딩 토큰의 인덱스(학습시 무시)
        # max_len : 입력 문장의 길이
        # dropout: dropout 비율

        super().__init__()
        self.pad_idx = pad_idx

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx = pad_idx)
        # 단어 임베딩: 단어 ID를 d_model 차원의 벡터로 변환
        self.pos_encoder = PositionalEncoding(d_model, max_len = max_len)
        # 위치 인코딩: 단어의 순서 정보를 벡터에 추가
        self.dropout = nn.Dropout(dropout)
        

        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        # 인코더 블럭 쌓기: TransformerEncoderBlock을 num_layer만큼 리스트로 생성
        ## nn.ModuleList를 써야 파이토치가 내부 layer들을 인식하고 추적할 수 있음.
    def forward(self, src_ids, src_mask = None):
        # src_ids: (batch_size, seq_len) 크기의 입력 단어 ID들
        # src_mask: 패딩 마스크(선택 사항)
        ## 임베딩 및 위치 정보 입력
        x = self.embedding(src_ids)    #(batch_size, seq_len) -> (batch_size, seq_len,d_model)
        x = self.pos_encoder(x)        # 위치 정보 추가
        x = self.dropout(x)            # dropout 적용

        ## 여러 개의 인코더 블록을 순차적으로 통과
        all_attn = [] # 각 층의 어텐션 가중치를 시각화 등을 통해 저장
        for layer in self.layers:
            #x는 다음 층의 입력이 됨(Hidden state의 업데이트)
            x, attn = layer(x, src_mask)
            all_attn.append(attn)
        return x, all_attn
        #최종 출력: (마지막 층의 출력 벡터, 모든 층의 어텐션 가중치 리스트)



In [17]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx, max_len, dropout=0.1):
        super().__init__()
        self.pad_idx = pad_idx

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx = pad_idx)
        self.pos_encoder = PositionalEncoding(d_model, max_len = max_len)
        self.dropout = nn.Dropout(dropout)
        

        self.layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, tgt_ids, enc_output, tgt_mask = None, src_tgt_mask = None):
        x = self.embedding(tgt_ids)   
        x = self.pos_encoder(x)        
        x = self.dropout(x)   

        all_self_attn = [] 
        all_cross_attn = []

        for layer in self.layers:
            x, self_attn, cross_attn = layer(x,enc_output, tgt_mask, src_tgt_mask)
            all_self_attn.append(self_attn)
            all_cross_attn.append(cross_attn)
        return x, all_self_attn,all_cross_attn

In [18]:
# source mask 만들기
## source에서 PAD를 가리기 위한 mask
def make_src_mask(src_ids, pad_idx = PAD_IDX):
    return (src_ids !=pad_idx).unsqueeze(0).unsqueeze(0)
    
# target padding mask
def make_tgt_padding_mask(tgt_ids, pad_idx = PAD_IDX):
    return (tgt_ids !=pad_idx).unsqueeze(1).unsqueeze(2)
    
# casual mask
## decoder는 미래 토큰을 보면 안되므로 lower triangular mask(하삼각 행렬)가 필요
def mask_causal_mask(tgt_ids):
    batch_size, tgt_len = tgt_ids.shape
    causal = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt_ids.device)).bool()
    return causal.unsqueeze(1).unsqueeze(2)

# decoder self-attneiton mask                                                                                                                                                                                                  
## padding mask와 casual mask를 합침
def make_tgt_mask(tgt_ids, pad_idx = PAD_IDX):
    tgt_padding_mask = make_tgt_padding_mask(tgt_ids, pad_idx)
    causal_mask = mask_causal_mask(tgt_ids)
    return tgt_padding_mask & causal_mask
    
# cross-attention용 mask
# decoder query가 source를 볼때, source의 PAD를 가리킴
def make_src_tgt_mask(src_ids, tgt_ids, pad_idx = PAD_IDX):
    src_valid = (src_ids != pad_idx).unsqueeze(1).unsqueeze(2)
    tgt_len = tgt_ids.size(1)
    return src_valid.expand(-1,1,tgt_len,-1)

In [19]:
 # Full Transformer

class Transformer(nn.Module):
    def __init__(
            self,
            vocab_size,
            d_model,
            num_heads,
            d_ff,
            num_encoder_layers,
            num_decoder_layers,
            pad_idx,
            max_len,
            dropout=0.1
    ):
        super().__init__()
        self.pad_idx = pad_idx

        self.encoder = TransformerEncoder(
            vocab_size = vocab_size,
            d_model = d_model,
            num_heads = num_heads,
            d_ff = d_ff,
            num_layers = num_encoder_layers,
            pad_idx = pad_idx,
            max_len = max_len,
            dropout = dropout
        )
        self.decoder = TransformerDecoder(
            vocab_size = vocab_size,
            d_model = d_model,
            num_heads = num_heads,
            d_ff = d_ff,
            num_layers = num_decoder_layers,
            pad_idx = pad_idx,
            max_len = max_len,
            dropout = dropout
        )

        self.output_proj = nn.Linear(d_model, vocab_size)

    def forward(self, src_ids, tgt_ids):
        src_mask = make_src_mask(src_ids, self.pad_idx)
        tgt_mask = make_tgt_mask(tgt_ids, self.pad_idx)
        src_tgt_mask = make_src_tgt_mask(src_ids, tgt_ids, self.pad_idx)

        enc_output, enc_attn = self.encoder(src_ids, src_mask)

        dec_output, dec_self_attn, dec_cross_attn = self.decoder(
            tgt_ids,
            enc_output,
            tgt_mask = tgt_mask,
            src_tgt_mask = src_tgt_mask
        )

        logits = self.output_proj(dec_output)

        return logits, enc_attn, dec_self_attn, dec_cross_attn

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = Transformer(
    vocab_size=len(vocab),
    d_model=64,
    num_heads=4,
    d_ff=128,
    num_encoder_layers=2,
    num_decoder_layers=2,
    pad_idx=PAD_IDX,
    max_len=20,
    dropout=0.1
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

Transformer(
  (encoder): TransformerEncoder(
    (embedding): Embedding(12, 64, padding_idx=0)
    (pos_encoder): PositionalEncoding()
    (dropout): Dropout(p=0.1, inplace=False)
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderBlock(
        (mha): MultiHeadAttention(
          (W_q): Linear(in_features=64, out_features=64, bias=False)
          (W_k): Linear(in_features=64, out_features=64, bias=False)
          (W_v): Linear(in_features=64, out_features=64, bias=False)
          (out_proj): Linear(in_features=64, out_features=64, bias=False)
        )
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=64, out_features=128, bias=True)
            (1): ReLU()
            (2): Dropout(p=0.1, inplace=False)
            (3): Linear(in_features=128, out_features=64, bias=True)
          )
        )
        (norm2): 

In [21]:
# 평가 함수
def evalute(model, loader):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for src_ids, dec_input_ids, dec_targbet_ids in loader:
            src_ids = src_ids.to(device)
            dec_input_ids = dec_input_ids.to(device)
            dec_targbet_ids = dec_targbet_ids.to(device)

            logits, _,_,_ = model(src_ids, dec_input_ids)

            loss = criterion(
                logits.view(-1, logits.size(-1)),
                dec_target_ids.view(-1)
            )

            total_loss += loss.item()
    return total_loss / len(loader)

### list comprehension


In [ ]:
import time

start = time.time()
list_2 = []
for x in range(100000):
    list_2.append(x)
end = time.time()
print(f"{end-start:.5f} sec")

# list comprehension을 사용하는 이유는 데이터 분석 시 속도 측면에서 사용함(권장하는 문법은 아님)

import time

start = time.time()
lc_1 = [x for x in range(100000)]

end = time.time()
print(f"{end-start:.5f} sec")